# 53. `hessian="jax"` vs `hessian="numerical"`

**Objectives:**

- Fit the same model with `hessian="numerical"` (default) and `hessian="jax"`.
- Compare the resulting parameter errors between the two Hessian modes.
- Note why `hessian="jax"` requires second-order differentiability.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2, and daughter
indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

## 1. Model and toy data

Same rho(770) + non-resonant model as the other lessons in this group, with a floatable
Cartesian coefficient on the non-resonant term.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=60, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

data = generate_toy(
    model, 1500, parameters=truth, seed=2026,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
session = FitSession(model, data)
print(f"Generated {data.size} unweighted events; truth = {truth}")

Generated 1500 unweighted events; truth = {'NR.x': 0.55, 'NR.y': 0.3}


## 2. Fit with both Hessian modes

Per the `Minimizer` class docstring, `hessian="jax"` supplies automatic second derivatives to
Minuit, including during MIGRAD's internal HESSE calls, instead of Minuit's own finite-difference
Hessian (the `hessian="numerical"` default). This requires the objective to be differentiable
twice, which every lineshape/normalization path in this package is built to support.

In [3]:
start = {"NR.x": 0.35, "NR.y": 0.45}

result_numerical = session.fit(start, strategy=2, hesse=True, hessian="numerical")
result_jax = session.fit(start, strategy=2, hesse=True, hessian="jax")

print("hessian=numerical:", {n: round(float(result_numerical.values[n]), 4) for n in result_numerical.parameters})
print("hessian=jax      :", {n: round(float(result_jax.values[n]), 4) for n in result_jax.parameters})

hessian=numerical: {'NR.x': 0.6057, 'NR.y': 0.3122}
hessian=jax      : {'NR.x': 0.6057, 'NR.y': 0.3122}


## 3. Compare the parameter errors

The fitted central values should already agree closely (both modes are converging to the same
MIGRAD minimum); this compares the HESSE *errors*, which is the actual output the two Hessian
computations differ on.

In [4]:
for name in result_numerical.parameters:
    err_numerical = float(result_numerical.errors[name])
    err_jax = float(result_jax.errors[name])
    rel_diff = abs(err_jax - err_numerical) / max(abs(err_numerical), 1e-12)
    print(f"{name}: error(numerical)={err_numerical:.5f}  error(jax)={err_jax:.5f}  "
          f"rel diff={rel_diff:.2%}")
    assert rel_diff < 0.05, "hessian=jax and hessian=numerical errors disagree unexpectedly"
print("Errors agree closely between the two Hessian modes.")

NR.x: error(numerical)=0.02785  error(jax)=0.02785  rel diff=0.00%
NR.y: error(numerical)=0.02606  error(jax)=0.02606  rel diff=0.00%
Errors agree closely between the two Hessian modes.


## Summary and exercises

1. Time both fits (e.g. with `%timeit`) on a larger model to see when `hessian="jax"` is worth
   using over Minuit's numerical Hessian.
2. Pass `hessian="jax"` to `FitSession.minimizer(...)` directly when working with the low-level
   `Minimizer` API instead of `FitSession.fit`.
3. Try a model with a component whose lineshape is not twice-differentiable in JAX (if any) and
   observe the failure mode `hessian="jax"` produces.

Return to the [course guide](TUTORIALS.md).